Arrivals:
     Time  Quantity  Product     Type
0       0         1        0  Arrival
1       0         1        1  Arrival
2       1         1        0  Arrival
3       2         1        0  Arrival
4       3         1        1  Arrival
..    ...       ...      ...      ...
195   146         1        0  Arrival
196   147         1        1  Arrival
197   147         1        0  Arrival
198   148         1        0  Arrival
199   149         1        0  Arrival

[200 rows x 4 columns]

Demands:
     Time  Quantity  Product    Type
0       0         1        0  Demand
1       0         1        1  Demand
2       1         1        0  Demand
3       2         1        0  Demand
4       3         1        1  Demand
..    ...       ...      ...     ...
195   146         1        0  Demand
196   147         1        1  Demand
197   147         1        0  Demand
198   148         1        0  Demand
199   149         1        0  Demand

[200 rows x 4 columns]

Total Demand for each Product:


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from IPython.display import HTML
import os
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
class BatchProcessingMachine:
    def __init__(self, processing_times, batch_size):
        self.processing_times = processing_times
        self.batch_size = batch_size
        self.current_product = None
        self.finish_time = 0
        self.busy = False
        self.event_log = []

    def process(self, product, current_time):
        self.busy = True
        self.finish_time = current_time + self.processing_times[product]
        self.current_product = product
        self.event_log.append((self.current_product, current_time, self.finish_time, self.batch_size))
        return self.finish_time

    def update(self, current_time):
        if current_time >= self.finish_time and self.busy:
            self.busy = False
            return self.current_product
        return False
    
    def print_event_log(self, machine_name):
        for event in self.event_log:
            print(f"At machine {machine_name}: Batch of Product {event[0]} processed from {event[1]} to {event[2]}")

In [3]:
class Machine:
    def __init__(self, processing_times, setup_times):
        self.processing_times = processing_times
        self.setup_times = setup_times
        self.current_product = None
        self.finish_time = 0
        self.busy = False
        self.event_log = []

    def process(self, product, current_time):
        if self.current_product is not None:
            setup_time = self.setup_times[self.current_product][product]
        else:
            setup_time = self.setup_times[None][product]  # Use the setup time from 'None' to the new product
        self.current_product = product
        start_time = current_time + setup_time
        self.finish_time = start_time + self.processing_times[product]
        self.busy = True
        self.event_log.append((product, current_time, start_time, self.finish_time))
        return self.finish_time

    def update(self, current_time):
        if current_time >= self.finish_time and self.busy:
            self.busy = False
            return self.current_product
        return None

    def print_event_log(self, machine_name):
        for event in self.event_log:
            print(f"At machine {machine_name}: Product {event[0]} processed from {event[1]} to {event[3]}")

queue0 Event Log:
Time 0: Queue Sizes - Product 0: 1, Product 1: 0
Time 0: Queue Sizes - Product 0: 1, Product 1: 1
Time 1: Queue Sizes - Product 0: 2, Product 1: 1
Time 2: Queue Sizes - Product 0: 3, Product 1: 1
Time 3: Queue Sizes - Product 0: 3, Product 1: 2
Time 3: Queue Sizes - Product 0: 4, Product 1: 2
Time 3: Queue Sizes - Product 0: 0, Product 1: 2
Time 4: Queue Sizes - Product 0: 1, Product 1: 2
Time 5: Queue Sizes - Product 0: 2, Product 1: 2
Time 6: Queue Sizes - Product 0: 3, Product 1: 2
Time 6: Queue Sizes - Product 0: 3, Product 1: 3
Time 7: Queue Sizes - Product 0: 4, Product 1: 3
Time 8: Queue Sizes - Product 0: 5, Product 1: 3
Time 9: Queue Sizes - Product 0: 6, Product 1: 3
Time 9: Queue Sizes - Product 0: 6, Product 1: 4
Time 10: Queue Sizes - Product 0: 7, Product 1: 4
Time 11: Queue Sizes - Product 0: 8, Product 1: 4
Time 12: Queue Sizes - Product 0: 9, Product 1: 4
Time 12: Queue Sizes - Product 0: 9, Product 1: 5
Time 13: Queue Sizes - Product 0: 10, Product 1

In [ ]:
import matplotlib.pyplot as plt

# Plotting queue sizes including 'queue_fin'
fig, axes = plt.subplots(6, 1, figsize=(10, 15), sharex=True)  # Share x-axis across all subplots and increase to 5 subplots
fig.suptitle('Queue Sizes Over Time', fontsize=16)

queue_names = ['queue0', 'queue1', 'queue2', 'queue3', 'queue_fin','demand']
colors = ['blue', 'green']
labels = ['Product 0', 'Product 1']

# Determine the global minimum and maximum time across all queues for consistent x-axis
all_times = [log[1] for queue in queue_names for log in production_line.logs[queue]]
global_min_time = min(all_times)
global_max_time = max(all_times)

for i, ax in enumerate(axes):
    for product_type in range(2):
        times = [log[1] for log in production_line.logs[queue_names[i]]]
        sizes = [log[0][product_type] for log in production_line.logs[queue_names[i]]]
        ax.plot(times, sizes, color=colors[product_type], label=f'Product {product_type}')
    ax.set_title(f'{queue_names[i]}')
    ax.set_xlim([global_min_time, global_max_time])  # Set consistent x-axis range
    ax.set_xlabel('Time')
    ax.set_ylabel('Number of Units')
    ax.legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Setup for Gantt chart plotting using matplotlib
fig, axs = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Gantt Chart for Production Line')

# Color coding for products and setups
colors = {0: 'skyblue', 1: 'lightgreen', 'setup': 'grey'}

machines = ['machine0', 'machine1', 'machine2', 'machine3']
machine_names = ['Batch Machine 0', 'Machine 1', 'Batch Machine 2', 'Machine 3']

# Plot each machine's activities
for ax, machine_name, name in zip(axs, machines, machine_names):
    machine = getattr(production_line, machine_name)
    y = 0
    for log in machine.event_log:
        if len(log) == 4:  # BatchProcessingMachine or Machine with setup
            product, current_time, start_time, finish_time = log
            # Calculate setup time and process time
            setup_time = start_time - current_time
            process_time = finish_time - start_time
            # Plot setup time if exists
            if setup_time > 0:
                ax.broken_barh([(current_time, setup_time)], (y, 0.8), facecolors='grey', edgecolor='black', linewidth=0.5)
            # Plot process time
            ax.broken_barh([(start_time, process_time)], (y, 0.8), facecolors=colors[product], edgecolor='black', linewidth=0.5)
        else:  # For single product processing without batch
            product, current_time, finish_time, _ = log
            process_time = finish_time - current_time
            ax.broken_barh([(current_time, process_time)], (y, 0.8), facecolors=colors[product], edgecolor='black', linewidth=0.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_yticks([0])
    ax.set_yticklabels([name])
    ax.grid(True)

# Create legend for product colors
patches = [mpatches.Patch(color=colors[key], label=f'Product {key}' if key != 'setup' else 'Setup Time') for key in colors]
fig.legend(handles=patches, loc='upper right')

plt.xlabel('Time')
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for title
plt.show()

In [ ]:
#Next steps
# 1) Introduce a demand forecast into the state space and reward it for meeting future excess demand
# 2) See if it can switch demand patterns quickly 
# 3) Q-learning implementaion to achive the same result
# 4) Write a LP program of this

In [ ]:
queues = {'queue0': [[], []], 'queue1': [[], []], 'queue2': [[], []], 'queue3': [[], []],'queue_fin': [[], []],'demand': [[10,12], [23,45,54]]}
print(len(queues['demand'][0])+len(queues['demand'][1]))
for product_type in range(2):
    print(product_type)